# PACE Phase 11E3 — Official AMPGAN v2 preprocessing-source audit

**Purpose:** Diagnose missing `data/dbaasp/clean.csv` without inventing training labels, changing PACE manuscript, or downloading checkpoints again. If the v2.0 source folder already exists, it is reused. This notebook *does not* attempt generation yet.

Upload the resulting `PACE_Phase11E3_Source_Diagnostics.zip` in ChatGPT.

In [ ]:
from pathlib import Path
SOURCE = '"""Analysis-only Phase 11E3: audit official AMPGAN v2 preprocessing dependencies.\n\nAll source files come from a commit-pinned official GitLab checkout. No synthetic\ntraining labels, invented condition vectors, or changes to the original PACE data.\nThis does not claim to generate peptides; it collects source needed for a faithful fix.\n"""\nfrom __future__ import annotations\nimport ast, hashlib, json, os, re, shutil, subprocess, sys, traceback, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nROOT = Path(\'/content\') if Path(\'/content\').exists() else Path.cwd()\nREPO = ROOT / \'ampgan_v2_official\'\nOUT = ROOT / \'PACE_Phase11E3_Source_Diagnostics\'\nOUT.mkdir(parents=True, exist_ok=True)\nZIP = ROOT / \'PACE_Phase11E3_Source_Diagnostics.zip\'\nCOMMIT = \'2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\'\nURL = \'https://gitlab.com/vail-uvm/amp-gan.git\'\nSOURCES = [\'ampgan/dbaasp.py\', \'ampgan/avpdb.py\', \'ampgan/uniprot.py\', \'ampgan/data_utils.py\', \'ampgan/generate_samples.py\', \'ampgan/model.py\', \'ampgan/README.md\', \'scripts/get_dbaasp_data.sh\']\nTARGETS = [\'data/dbaasp/clean.csv\',\'data/avpdb/clean.csv\',\'data/uniprot/clean.csv\', \'results/mic50_bins.json\',\'results/target_groups_values.json\',\'results/targets_values.json\']\nSTATUS={\'phase\':\'11E3\',\'scope\':\'official source preprocessing only\',\'timestamp_utc\':datetime.now(timezone.utc).isoformat(),\n        \'official_repo\':URL,\'required_commit\':COMMIT,\'auto_preprocessing\':False,\n        \'generated_peptides\':0,\'inputs_untouched\':True,\'files\':{},\'raw_data\':{},\'functions\':{},\'targets\':{},\'warnings\':[]}\n\ndef run(*args,cwd=None,timeout=60):\n    p=subprocess.run(args,cwd=cwd,timeout=timeout,capture_output=True,text=True)\n    return {\'args\':list(args),\'returncode\':p.returncode,\'stdout\':p.stdout[-2000:],\'stderr\':p.stderr[-2000:]}\n\ndef sha(p):\n    h=hashlib.sha256()\n    with p.open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\ndef describe_python(path):\n    src=path.read_text(\'utf-8\',errors=\'replace\')\n    tree=ast.parse(src, filename=str(path))\n    functions=[]; main_blocks=[]\n    for node in tree.body:\n        if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)):\n            args=[a.arg for a in node.args.args]\n            defaults=len(node.args.defaults)\n            functions.append({\'name\':node.name,\'line\':node.lineno,\'arguments\':args,\'required_positional\':args[:len(args)-defaults],\n                              \'contains_to_csv\':(\'to_csv(\' in ast.get_source_segment(src,node) if ast.get_source_segment(src,node) else False),\n                              \'contains_clean_path\':(\'clean.csv\' in ast.get_source_segment(src,node) if ast.get_source_segment(src,node) else False)})\n        if isinstance(node,ast.If) and isinstance(node.test,ast.Compare):\n            s=ast.get_source_segment(src,node.test) or \'\'\n            if \'__name__\' in s and \'__main__\' in s:\n                main_blocks.append({\'line\':node.lineno,\'source\':(ast.get_source_segment(src,node) or \'\')[:6000]})\n    return {\'functions\':functions,\'main_blocks\':main_blocks,\n            \'path_references\':sorted(set(re.findall(r\'\\.\\./(?:data|results)/[A-Za-z0-9_./-]+\',src)))[:140],\n            \'csv_writes\':len(re.findall(r\'\\.to_csv\\(\',src))}\n\ndef run_audit():\n    if not REPO.is_dir():\n        env=os.environ.copy();env[\'GIT_LFS_SKIP_SMUDGE\']=\'1\'\n        p=subprocess.run([\'git\',\'clone\',\'--depth\',\'1\',\'--branch\',\'v2.0\',URL,str(REPO)],env=env,capture_output=True,text=True,timeout=300)\n        if p.returncode:raise RuntimeError(\'Official source clone failed: \'+p.stderr[-1000:])\n        STATUS[\'cloned_source_only\']=True\n    head=run(\'git\',\'rev-parse\',\'HEAD\',cwd=REPO)\n    STATUS[\'commit\']=head[\'stdout\'].strip()\n    if STATUS[\'commit\'] != COMMIT:raise RuntimeError(\'Pinned original commit mismatch; refusing to inspect unrelated checkout: \'+STATUS[\'commit\'])\n    for rel in SOURCES:\n        p=REPO/rel\n        if not p.is_file():\n            STATUS[\'warnings\'].append(\'Missing official file: \'+rel);continue\n        target=OUT/\'official_source\'/rel;target.parent.mkdir(exist_ok=True,parents=True)\n        shutil.copyfile(p,target)\n        STATUS[\'files\'][rel]={\'size\':p.stat().st_size,\'sha256\':sha(p)}\n        if p.suffix==\'.py\':\n            try: STATUS[\'functions\'][rel]=describe_python(p)\n            except Exception as e:STATUS[\'warnings\'].append(rel+\' AST: \'+repr(e))\n    for rel in TARGETS:\n        p=REPO/rel\n        STATUS[\'targets\'][rel]={\'exists\':p.is_file(),\'bytes\':p.stat().st_size if p.is_file() else None}\n    # Capture only minimal RAW-STRUCTURE excerpts, never full DBAASP JSON or 47 MB UniProt.\n    for rel in [\'data/dbaasp/raw.json\',\'data/avpdb/AVPdb_data.tsv\',\'data/avpdb/mAVPdb_data.tsv\',\'data/uniprot/uniprot.tab\']:\n        p=REPO/rel\n        info={\'exists\':p.is_file(),\'size_bytes\':p.stat().st_size if p.is_file() else None}\n        if p.is_file():\n            with p.open(\'rb\') as f:prefix=f.read(1800)\n            if prefix.startswith(b\'version https://git-lfs.github.com/spec/v1\'):\n                info[\'git_lfs_pointer\']=True\n            else:\n                info[\'git_lfs_pointer\']=False\n                snippet=prefix.decode(\'utf-8\',errors=\'replace\')[:1000]\n                (OUT/(\'preview_\'+rel.replace(\'/\',\'_\')+\'.txt\')).write_text(snippet,encoding=\'utf-8\')\n        STATUS[\'raw_data\'][rel]=info\n    # Check whether the original code defines a preprocessing entry point. Do not\n    # invoke arbitrary functions or edit source; user can share source for review.\n    source_funcs=STATUS[\'functions\'].get(\'ampgan/dbaasp.py\',{})\n    STATUS[\'preprocessing_candidates\']=[f for f in source_funcs.get(\'functions\',[]) if any(term in f[\'name\'].lower() for term in (\'clean\',\'process\',\'parse\',\'save\',\'load\',\'prepar\'))]\n    STATUS[\'status\']=\'source_captured_no_inference\'\n\ndef package():\n    (OUT/\'status.json\').write_text(json.dumps(STATUS,indent=2,default=str))\n    (OUT/\'README.txt\').write_text(\'Pinned GitLab v2.0 source diagnostic for missing AMPGAN training preprocessing files.\\n\'\n       \'NO PEPTIDES GENERATED. NO TRAINING SOURCES OR MODEL CHECKPOINT MODIFIED.\\n\'\n       \'Upload ZIP for inspection of original dbaasp.py, avpdb.py, uniprot.py and data_utils.py.\\n\')\n    with zipfile.ZipFile(ZIP,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for f in sorted(OUT.rglob(\'*\')):\n            if f.is_file():z.write(f,f.relative_to(OUT))\n    with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None\n    print(\'DIAGNOSTIC STATUS:\',STATUS[\'status\'])\n    print(\'Official commit:\',STATUS.get(\'commit\'))\n    print(\'Missing outputs:\',[p for p,d in STATUS[\'targets\'].items() if not d[\'exists\']])\n    print(\'DATA:\',json.dumps(STATUS[\'raw_data\'],indent=2))\n    print(\'OUTPUT ZIP:\',ZIP,\'bytes\',ZIP.stat().st_size)\n\nif __name__==\'__main__\':\n    try:run_audit()\n    except BaseException as e:\n        STATUS[\'status\']=\'diagnostic_error\'\n        STATUS[\'error\']=repr(e)\n        (OUT/\'error_traceback.txt\').write_text(traceback.format_exc())\n    finally:package()\n'
Path("/content/PACE_Phase11E3_Official_Preprocessing_Diagnostic.py").write_text(SOURCE)
exec(compile(SOURCE,"PACE_Phase11E3_Official_Preprocessing_Diagnostic.py","exec"),{"__name__":"__main__"})

In [ ]:
from google.colab import files
files.download("/content/PACE_Phase11E3_Source_Diagnostics.zip")